In [1]:
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Deep Learning libraries

import numpy as np
from tensorflow.keras import layers, models

In [4]:
# Load preprocessed data

X = np.load("../data/X.npy").astype(np.float32)
Y = np.load("../data/Y.npy").astype(np.float32)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (3933, 256, 256, 3)
Y shape: (3933, 256, 256)


In [5]:
# Add channel dimension to masks

Y = np.expand_dims(Y, axis=-1)

print("New Y shape:", Y.shape)

New Y shape: (3933, 256, 256, 1)


In [6]:
# Input layer 
#reduced model is taken actually during training

inputs = layers.Input(shape=(256, 256, 3))

In [7]:
# Encoder Block 1

c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c1) # c1= (256,256,64)
p1 = layers.MaxPooling2D((2,2))(c1)
#outputp1 will be (128,128,64) that is after pooling

In [8]:
# Encoder Block 2

c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(p1)
c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c2) #c2 = (128,128,128)
p2 = layers.MaxPooling2D((2,2))(c2)
#outputp2 will be (64,64,128) that is after pooling

In [9]:
# Encoder Block 3

c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(p2)
c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c3) #c3 = (64,64,256)
p3 = layers.MaxPooling2D((2,2))(c3)
#outputp3 will be (32,32,256) that is after pooling

In [10]:
# Encoder Block 4

c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(p3)
c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(c4) #c4 = (32,32,512)
p4 = layers.MaxPooling2D((2,2))(c4)
#outputp4 = (16,16,512) that is after pooling

In [11]:
# Bottleneck

c5 = layers.Conv2D(1024, (3,3), activation='relu', padding='same')(p4)
c5 = layers.Conv2D(1024, (3,3), activation='relu', padding='same')(c5) # c5 = (16,16,1024)
# here c5 is now (16,16,1024) and no pooling here since We already reached smallest size,next step is upsampling decoder begins

In [12]:
# Decoder Block 1

u6 = layers.UpSampling2D((2,2))(c5)
u6 = layers.concatenate([u6, c4]) # total featues after concat is (32,32,1024) + (32,32,512) = (32,32,1536)

c6 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(u6)
c6 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(c6) #c6 = (32,32,512)

In [13]:
# Decoder Block 2

u7 = layers.UpSampling2D((2,2))(c6)
u7 = layers.concatenate([u7, c3]) # total featuers after concat is (64,64,512) + (64,64,256) = (64,64,768)

c7 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(u7)
c7 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c7) # c7 =(64,64,256)

In [14]:
# Decoder Block 3

u8 = layers.UpSampling2D((2,2))(c7)
u8 = layers.concatenate([u8, c2]) # total features after concat is (128,128,256) + (128,128,128) = (128,128,384)

c8 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(u8)
c8 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c8) #c8 = (128,128,128)

In [15]:
# Decoder Block 4

u9 = layers.UpSampling2D((2,2))(c8)
u9 = layers.concatenate([u9, c1]) # total features after concat is (256,256,128) + (256,256,64) = (256,256,192)

c9 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(u9)
c9 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c9) # c9 = (256,256,64)

In [16]:
# Output Layer

outputs = layers.Conv2D(1, (1,1), activation='sigmoid')(c9)

In [17]:
# Create Model

model = models.Model(inputs, outputs)

In [18]:
# Model Summary

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 256, 256, 64  1792        ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 conv2d_1 (Conv2D)              (None, 256, 256, 64  36928       ['conv2d[0][0]']                 
                                )                                                             

In [19]:
# ── Custom Loss & Metrics ───────────────────────────────────────────

import tensorflow as tf

# Dice Coefficient
def dice_coef(y_true, y_pred, smooth=1e-6):# y_true = actual mask from dataset 0 and 1, y_pred =what our model predicts float value 0.0 and 1.0, 1e-6 add an tiny no. 0.000001 to prevent divison by 0. 
    y_true_f = tf.keras.backend.flatten(tf.cast(y_true, tf.float32))#tf.cast(y_true, tf.float32) → converts mask from int (0,1) to float (0.0, 1.0) because math operations need float and flatten convert 2d mask image into 1d array
    y_pred_f = tf.keras.backend.flatten(y_pred)#Same flattening for predictions. Already float since sigmoid outputs floats.

    intersection = tf.keras.backend.sum(y_true_f * y_pred_f) #Multiplies the two flattened arrays element by element, then sums everything.This counts how many pixels both agree are tumor.

    return (2. * intersection + smooth) / ( # this is dice coeff formula
        tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth
    )

# Dice Loss
def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred) #dice loss = 1 - dice_coef

# BCE + Dice Loss
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred) # this calculates combined loss = BCE + Dice loss

# IoU Metric
def iou(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = tf.keras.backend.flatten(y_pred)

    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)

    union = (
        tf.keras.backend.sum(y_true_f)
        + tf.keras.backend.sum(y_pred_f)
        - intersection
    )

    return (intersection + smooth) / (union + smooth) # iou formula

In [20]:
# ── Compile Model ───────────────────────────────────────────────────

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=[dice_coef, iou, 'accuracy']
)